In [ ]:
import json
from pathlib import Path
from typing import Literal

import numpy as np
import polars as pl
from datasets import Dataset, DatasetDict, load_dataset

In [ ]:
DATA_DIR = "../data"
DATASET_NAME = "yambda"

OUTPUT_DIR = f"{DATA_DIR}/{DATASET_NAME}"
DATASET_PATH = f"{DATA_DIR}/{DATASET_NAME}/raw"

Path(DATASET_PATH).mkdir(parents=True, exist_ok=True)

NUM_PARTS = 10
EMBEDDINGS_PATH = f"{DATASET_PATH}/embeddings.parquet"

Download dataset with embeddings

In [ ]:
!hf download yandex/yambda embeddings.parquet --repo-type dataset --local-dir ../data/yambda/raw

In [ ]:
class YambdaDataset:
    INTERACTIONS = frozenset(["likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"])

    def __init__(
        self, dataset_type: Literal["flat", "sequential"] = "flat", dataset_size: Literal["50m", "500m", "5b"] = "50m"
    ):
        assert dataset_type in {"flat", "sequential"}
        assert dataset_size in {"50m", "500m", "5b"}
        self.dataset_type = dataset_type
        self.dataset_size = dataset_size

    def interaction(
        self, event_type: Literal["likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"]
    ) -> Dataset:
        assert event_type in YambdaDataset.INTERACTIONS
        return self._download(f"{self.dataset_type}/{self.dataset_size}", event_type)

    def audio_embeddings(self) -> Dataset:
        return self._download("", "embeddings")

    def album_item_mapping(self) -> Dataset:
        return self._download("", "album_item_mapping")

    def artist_item_mapping(self) -> Dataset:
        return self._download("", "artist_item_mapping")

    @staticmethod
    def _download(data_dir: str, file: str) -> Dataset:
        data = load_dataset("yandex/yambda", data_dir=data_dir, data_files=f"{file}.parquet")
        assert isinstance(data, DatasetDict)
        return data["train"]


dataset = YambdaDataset("flat", "50m")
likes = dataset.interaction("likes")

In [ ]:
item_ids = pl.read_parquet(EMBEDDINGS_PATH)["item_id"].to_numpy()
all_data_interactions = (
    likes.to_polars()
    .filter(pl.col("item_id").is_in(item_ids))
    .sort(["timestamp"])
    .with_row_index("original_order")
    .rename({"uid": "user_id"})
)
all_data_interactions.head()

Remap User IDs

In [ ]:
user_mapping = all_data_interactions.select(pl.col("user_id")).unique().sort("user_id").with_row_index("new_user_id")

all_data_interactions = (
    all_data_interactions.join(user_mapping, on="user_id", how="left")
    .with_columns(pl.col("new_user_id").cast(pl.Int64).alias("user_id"))
    .drop("new_user_id")
)
all_data_interactions.head()

Remap Item IDs

In [ ]:
all_data_items = all_data_interactions.select("item_id").unique()
all_data_users = all_data_interactions.select("user_id").unique()

unique_items_sorted = all_data_items.sort("item_id").with_row_index("new_item_id")
global_item_mapping = dict(zip(unique_items_sorted["item_id"], unique_items_sorted["new_item_id"], strict=True))

print(f"Total users: {all_data_users.shape[0]}, Total items: {len(global_item_mapping)}")

Save Item IDs Mapping

In [ ]:
mapping_output_path = f"{OUTPUT_DIR}/global_item_mapping.json"

with open(mapping_output_path, "w") as f:
    json.dump({str(k): v for k, v in global_item_mapping.items()}, f, indent=2)

print(f"Mapping saved: {mapping_output_path}")

Filter out Items without Embeddings

In [ ]:
item_ids = pl.read_parquet(EMBEDDINGS_PATH)["item_id"].to_numpy()
item_embeddings = pl.read_parquet(EMBEDDINGS_PATH)["normalized_embed"].to_numpy()

mask = np.isin(item_ids, all_data_items.to_numpy())

item_ids = item_ids[mask]
item_embeddings = np.array([x.tolist() for x in item_embeddings[mask]])

Save Item Embeddings

In [ ]:
items_metadata = pl.DataFrame({"item_id": item_ids, "embedding": item_embeddings})

# items_metadata.write_parquet(EMBEDDINGS_PATH) not needed since it's already saved
items_metadata.head()

Remap Interations Item IDs

In [ ]:
def remap_interactions(df, mapping):
    return df.with_columns(pl.col("item_id").replace_strict(mapping, return_dtype=pl.UInt32))


all_data_interactions_remapped = remap_interactions(all_data_interactions, global_item_mapping)
items_metadata_remapped = remap_interactions(items_metadata, global_item_mapping)

Assign Parts

In [ ]:
all_data_interactions_remapped_sorted = all_data_interactions_remapped.sort("original_order")
all_data_interactions_remapped_sorted = all_data_interactions_remapped_sorted.with_row_index("row_nr")

base_size = all_data_interactions_remapped_sorted.height // NUM_PARTS

all_data_interactions_with_groups = all_data_interactions_remapped_sorted.with_columns(
    (pl.col("row_nr") // (base_size + 1)).alias("part")
).drop("row_nr")

all_data_interactions_with_groups.head()

Check Stats

In [ ]:
parts_distribution = (
    all_data_interactions_with_groups.group_by("part")
    .agg(
        pl.count().alias("count"),
        pl.col("original_order").min().alias("min_order"),
        pl.col("original_order").max().alias("max_order"),
    )
    .sort("part")
)

print("Distribution by part:")
print(parts_distribution)

print(f"Minimum part: {parts_distribution['part'].min()}")
print(f"Maximum part: {parts_distribution['part'].max()}")
print(f"Total number of events from all parts: {parts_distribution['count'].sum()}")

Save Dataset

In [ ]:
def write_parquet(output_dir, data, file_name):
    print(f"Shape: {data.shape}")
    output_parquet_path = f"{output_dir}/{file_name}.parquet"
    data.write_parquet(output_parquet_path)
    print(f"File saved: {file_name}")


write_parquet(OUTPUT_DIR, items_metadata_remapped, "items_metadata_remapped")
write_parquet(OUTPUT_DIR, all_data_interactions_with_groups, "all_data_interactions_with_groups")